# LAB | Feature Engineering

**Load the data**

In this challenge, we will be working with the same Spaceship Titanic data, like the previous Lab. The data can be found here:

https://raw.githubusercontent.com/data-bootcamp-v4/data/main/spaceship_titanic.csv

Metadata

https://github.com/data-bootcamp-v4/data/blob/main/spaceship_titanic.md

In [17]:
#Libraries
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

In [18]:
spaceship = pd.read_csv("https://raw.githubusercontent.com/data-bootcamp-v4/data/main/spaceship_titanic.csv")
spaceship.head()

,PassengerId,HomePlanet,CryoSleep,Cabin,Destination,Age,VIP,RoomService,FoodCourt,ShoppingMall,Spa,VRDeck,Name,Transported
0,0001_01,Europa,False,B/0/P,TRAPPIST-1e,39.0,False,0.0,0.0,0.0,0.0,0.0,Maham Ofracculy,False
1,0002_01,Earth,False,F/0/S,TRAPPIST-1e,24.0,False,109.0,9.0,25.0,549.0,44.0,Juanna Vines,True
2,0003_01,Europa,False,A/0/S,TRAPPIST-1e,58.0,True,43.0,3576.0,0.0,6715.0,49.0,Altark Susent,False
3,0003_02,Europa,False,A/0/S,TRAPPIST-1e,33.0,False,0.0,1283.0,371.0,3329.0,193.0,Solam Susent,False
4,0004_01,Earth,False,F/1/S,TRAPPIST-1e,16.0,False,303.0,70.0,151.0,565.0,2.0,Willy Santantines,True


**Check the shape of your data**

In [19]:
print('shape of data is:',spaceship.shape)

shape of data is: (8693, 14)


**Check for data types**

In [20]:
print (spaceship.dtypes)

PassengerId      object
HomePlanet       object
CryoSleep        object
Cabin            object
Destination      object
Age             float64
VIP              object
RoomService     float64
FoodCourt       float64
ShoppingMall    float64
Spa             float64
VRDeck          float64
Name             object
Transported        bool
dtype: object


**Check for missing values**

In [21]:
print('sum of null values in rows per column:\n',spaceship.isnull().sum())

sum of null values in rows per column:
 PassengerId       0
HomePlanet      201
CryoSleep       217
Cabin           199
Destination     182
Age             179
VIP             203
RoomService     181
FoodCourt       183
ShoppingMall    208
Spa             183
VRDeck          188
Name            200
Transported       0
dtype: int64


There are multiple strategies to handle missing data

- Removing all rows or all columns containing missing data.
- Filling all missing values with a value (mean in continouos or mode in categorical for example).
- Filling all missing values with an algorithm.

For this exercise, because we have such low amount of null values, we will drop rows containing any missing value. 

In [22]:
#dropping the null values and printing the sum to check
spaceship = spaceship.dropna()
print(spaceship.isna().sum())

PassengerId     0
HomePlanet      0
CryoSleep       0
Cabin           0
Destination     0
Age             0
VIP             0
RoomService     0
FoodCourt       0
ShoppingMall    0
Spa             0
VRDeck          0
Name            0
Transported     0
dtype: int64


- **Cabin** is too granular - transform it in order to obtain {'A', 'B', 'C', 'D', 'E', 'F', 'G', 'T'}

In [23]:
#transforming cabin: keeping only the deck letter (index zero); no missing values remain after dropna
spaceship['Cabin'] = spaceship['Cabin'].str.split('/').str[0]
spaceship['Cabin'].value_counts(dropna=False)

Cabin
F    2152
G    1973
E     683
B     628
C     587
D     374
A     207
T       2
Name: count, dtype: int64

- Drop PassengerId and Name

In [24]:
# dropping columns PassengerId and Name
spaceship = spaceship.drop(columns=['PassengerId', 'Name'])

**Perform Train Test Split**

We split **before** any encoding, imputing or scaling, so that these steps are learned from the training data only.

In [25]:
# 1. Separate features and target
y = spaceship["Transported"].astype(int) # target
X = spaceship.drop(columns="Transported") # features

# CryoSleep and VIP are True/False: turn them into text categories
for c in ["CryoSleep", "VIP"]:
    X[c] = X[c].map({True: "True", False: "False"})

# 2. Split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
print("Training observations:", len(X_train))
print("Test observations:", len(X_test))
print("Training features:", list(X_train.columns))

Training observations: 5284
Test observations: 1322
Training features: ['HomePlanet', 'CryoSleep', 'Cabin', 'Destination', 'Age', 'VIP', 'RoomService', 'FoodCourt', 'ShoppingMall', 'Spa', 'VRDeck']


**Define the column groups by data type**

KNN is distance based and needs all input data to be numerical. Numerical columns need scaling; non-numerical columns need dummies (one-hot encoding).

In [26]:
num_cols = X_train.select_dtypes(include="number").columns.tolist()
cat_cols = X_train.select_dtypes(exclude="number").columns.tolist()

print("Numerical columns:", num_cols)
print("Categorical columns:", cat_cols)

Numerical columns: ['Age', 'RoomService', 'FoodCourt', 'ShoppingMall', 'Spa', 'VRDeck']
Categorical columns: ['HomePlanet', 'CryoSleep', 'Cabin', 'Destination', 'VIP']


- For non-numerical columns, do dummies.

In [27]:
X_train_d = pd.get_dummies(X_train, columns=cat_cols, dtype=int)
X_test_d = pd.get_dummies(X_test, columns=cat_cols, dtype=int)
X_test_d = X_test_d.reindex(columns=X_train_d.columns, fill_value=0)

**Define a pipeline for each data type**

- Numerical: standardize (spending columns reach the thousands while `Age` is below 80, so without scaling they would dominate the Euclidean distances).
- Categorical: one-hot encode (this replaces `pd.get_dummies`). `handle_unknown="ignore"` means a category not seen in training becomes all zeros instead of causing an error.

In [31]:
from sklearn.preprocessing import StandardScaler


num_pipe = Pipeline([
    ("scale",  StandardScaler()),
])
preprocess = ColumnTransformer(
    [("num", num_pipe, num_cols)],
    remainder="passthrough",
)


**Model Selection**

In this exercise we will be using **KNN** as our predictive model. It is the last step of the full pipeline, after the preprocessing.

In [32]:
#initializing and fitting the full pipeline (preprocessing + Knn)
knn_pipe = Pipeline([
    ("preprocess", preprocess),
    ("knn", KNeighborsClassifier(n_neighbors=5)),
])

- Evaluate your model's performance. Comment it

In [33]:
# Fit and evaluate using the dummy versions of the data
knn_pipe.fit(X_train_d, y_train)
y_pred_test_knn = knn_pipe.predict(X_test_d)

print("Test accuracy:", accuracy_score(y_test, y_pred_test_knn))
print(confusion_matrix(y_test, y_pred_test_knn))
print(classification_report(y_test, y_pred_test_knn))

Test accuracy: 0.7692889561270801
[[494 162]
 [143 523]]
              precision    recall  f1-score   support

           0       0.78      0.75      0.76       656
           1       0.76      0.79      0.77       666

    accuracy                           0.77      1322
   macro avg       0.77      0.77      0.77      1322
weighted avg       0.77      0.77      0.77      1322



The KNN model reached a test accuracy of about 76.9% (1,017 of 1,322 test passengers correct), well above the 50% baseline for this balanced dataset. 
The confusion matrix shows similar numbers of false positives (162) and false negatives (143), and precision, recall and F1 are around 0.75 to 0.79 for both classes, 
so the model is not biased toward either one. Recall is slightly higher for the transported class (0.79),
 meaning the model finds a little more of the passengers who were actually transported, while precision is slightly higher for the not-transported class (0.78).